## Importando Bibliotecas e Carregando os Dados Processados

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

In [ ]:
caminho_atual = Path(".").resolve()
if caminho_atual.name == "notebooks":
    raiz_projeto = caminho_atual.parent
else:
    raiz_projeto = caminho_atual

pasta_processed = raiz_projeto / "data" / "processed"

df_train = pd.read_csv(pasta_processed / "train_scaled.csv")
df_val   = pd.read_csv(pasta_processed / "val_scaled.csv")
df_test  = pd.read_csv(pasta_processed / "test_scaled.csv")

colunas_features = [c for c in df_train.columns if c.startswith("b1_")]

X_train = df_train[colunas_features].values.astype(np.float32)
X_val   = df_val[colunas_features].values.astype(np.float32)
X_test  = df_test[colunas_features].values.astype(np.float32)

print(f"Treino (Saudável):    {X_train.shape[0]} amostras x {X_train.shape[1]} features")
print(f"Validação (Saudável): {X_val.shape[0]} amostras  x {X_val.shape[1]} features")
print(f"Teste (Falha):        {X_test.shape[0]} amostras x {X_test.shape[1]} features")

## 1. Definindo o Modelo: MLP Autoencoder
O Autoencoder comprime as **11 features** originais até um **espaço latente (gargalo) de 4 números** e tenta reconstruir as 11 features originais na saída.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

class MLPAutoencoder(nn.Module):
    def __init__(self, input_dim=11, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.GELU(),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.GELU(),
            nn.Linear(16, 32),
            nn.GELU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

modelo = MLPAutoencoder(input_dim=X_train.shape[1], latent_dim=4)
print(modelo)

## 2. Treinamento Não-Supervisionado (Apenas com Dados Saudáveis)

In [ ]:
criterio = nn.MSELoss()
otimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-4)

loader_treino = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=32, shuffle=True)

historico_treino = []
historico_val = []

epocas = 150
for epoca in range(epocas):
    modelo.train()
    perda_epoca = 0.0
    for (lote,) in loader_treino:
        otimizador.zero_grad()
        reconstruido = modelo(lote)
        loss = criterio(reconstruido, lote)
        loss.backward()
        otimizador.step()
        perda_epoca += loss.item() * len(lote)
    
    historico_treino.append(perda_epoca / len(X_train))
    
    # Avaliando erro de reconstrução na validação saudável
    modelo.eval()
    with torch.no_grad():
        val_recon = modelo(torch.from_numpy(X_val))
        val_loss = criterio(val_recon, torch.from_numpy(X_val)).item()
        historico_val.append(val_loss)

print(f"MSE Final no Treino:    {historico_treino[-1]:.4f}")
print(f"MSE Final na Validação: {historico_val[-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(historico_treino, label="Treino (Saudável)", color="#2980b9", lw=2)
plt.plot(historico_val, label="Validação (Saudável)", color="#27ae60", lw=2)
plt.title("Curva de Aprendizado do MLP Autoencoder (MSE)")
plt.xlabel("Épocas")
plt.ylabel("Erro de Reconstrução (MSE)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Cálculo do Erro de Reconstrução e Definição do Limiar (*Threshold*)

In [ ]:
modelo.eval()
with torch.no_grad():
    recon_train = modelo(torch.from_numpy(X_train)).numpy()
    recon_val   = modelo(torch.from_numpy(X_val)).numpy()
    recon_test  = modelo(torch.from_numpy(X_test)).numpy()

# Erro Quadrático Médio (MSE) por amostra (arquivo)
erro_train = np.mean((X_train - recon_train) ** 2, axis=1)
erro_val   = np.mean((X_val - recon_val) ** 2, axis=1)
erro_test  = np.mean((X_test - recon_test) ** 2, axis=1)

# Limiar definido como o Percentil 99 do erro no treino saudável
limiar = np.percentile(erro_train, 99)
print(f"Limiar de Anomalia (Percentil 99 do Treino): {limiar:.4f}")

## 4. Avaliação: Curva ROC, Matriz de Confusão e Detecção de Falha (RUL)

In [ ]:
# Juntando Validação (Saudável = 0) e Teste (Falha = 1) para calcular as métricas
y_real = np.concatenate([np.zeros(len(erro_val)), np.ones(len(erro_test))])
scores = np.concatenate([erro_val, erro_test])
y_pred = (scores > limiar).astype(int)

df_resultados = pd.DataFrame([{
    "Modelo": "MLP Autoencoder",
    "Accuracy": accuracy_score(y_real, y_pred),
    "Precision": precision_score(y_real, y_pred),
    "Recall (Taxa de Detecção)": recall_score(y_real, y_pred),
    "F1-Score": f1_score(y_real, y_pred),
    "ROC-AUC": roc_auc_score(y_real, scores)
}])
display(df_resultados.round(4))

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 5))

# 1. Curva ROC
fpr, tpr, _ = roc_curve(y_real, scores)
auc_val = roc_auc_score(y_real, scores)
axes[0].plot(fpr, tpr, lw=2.2, color="#27ae60", label=f"MLP Autoencoder (AUC = {auc_val:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("Curva ROC (Validação Saudável vs. Teste com Falha)", fontweight="bold")
axes[0].set_xlabel("Taxa de Falsos Positivos (FPR)")
axes[0].set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# 2. Matriz de Confusão
cm = confusion_matrix(y_real, y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
    xticklabels=["Saudável (0)", "Falha (1)"],
    yticklabels=["Saudável (0)", "Falha (1)"]
)
axes[1].set_title("Matriz de Confusão", fontweight="bold")
axes[1].set_xlabel("Predito pelo Autoencoder")
axes[1].set_ylabel("Estado Real")

plt.tight_layout()
plt.show()

In [ ]:
# Cálculo do 1º Alarme Sustentado (5 arquivos consecutivos acima do limiar) e RUL
acima_limiar = (erro_test > limiar).astype(int)
indices_teste = df_test["idx"].values
janela_sustentada = 5
fim_da_vida_idx = 981

primeiro_alarme_idx = None
for i in range(len(acima_limiar) - janela_sustentada + 1):
    if np.all(acima_limiar[i : i + janela_sustentada] == 1):
        primeiro_alarme_idx = int(indices_teste[i])
        break

rul_arquivos = fim_da_vida_idx - primeiro_alarme_idx
rul_horas = rul_arquivos * 10.0 / 60.0

print(f"1º Alarme Sustentado no Arquivo: {primeiro_alarme_idx}")
print(f"Vida Útil Restante (RUL) no Alarme: {rul_arquivos} arquivos (~{rul_horas:.2f} horas antes do colapso)")

In [ ]:
# Plotando a Evolução do Erro de Reconstrução ao longo de toda a vida do Rolamento
plt.figure(figsize=(13, 5))

plt.plot(df_train["idx"], erro_train, color="#27ae60", label="Treino (Saudável)", lw=1.5)
plt.plot(df_val["idx"], erro_val, color="#2980b9", label="Validação (Saudável)", lw=1.5)
plt.plot(df_test["idx"], erro_test, color="#c0392b", label="Teste (Degradação / Falha)", lw=1.5)

plt.axhline(limiar, color="black", linestyle="--", lw=1.5, label=f"Limiar = {limiar:.3f}")
if primeiro_alarme_idx is not None:
    plt.axvline(primeiro_alarme_idx, color="#e67e22", linestyle="-.", lw=2,
                label=f"1º Alarme (Idx {primeiro_alarme_idx} | RUL = {rul_horas:.1f}h)")

plt.yscale("log")
plt.title("Monitoramento do Rolamento 1 — Erro de Reconstrução do Autoencoder (Escala Log)")
plt.xlabel("Índice do Arquivo (10 min por arquivo)")
plt.ylabel("Erro de Reconstrução (MSE - log)")
plt.legend(loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Interpretação do Modelo: Contribuição de Cada Variável no Alarme (*Feature Importance*)

In [ ]:
# Qual variável mais aumentou o Erro de Reconstrução no momento do 1º alarme?
pos_alarme = np.where(indices_teste == primeiro_alarme_idx)[0][0]

erro_por_feature = (X_test[pos_alarme] - recon_test[pos_alarme]) ** 2
contribuicao_pct = (erro_por_feature / np.sum(erro_por_feature)) * 100

imp_ae = pd.Series(contribuicao_pct, index=colunas_features).sort_values(ascending=True)

plt.figure(figsize=(8, 4.5))
imp_ae.plot(kind="barh", color="#2980b9", edgecolor="black", alpha=0.85)
plt.title(f"Importância das Variáveis no 1º Alarme (Arquivo {primeiro_alarme_idx})", fontweight="bold")
plt.xlabel("Contribuição para o Erro de Reconstrução (%)")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Salvando o Modelo Final e os Scores de Predição

In [ ]:
pasta_reports = raiz_projeto / "notebooks" / "reports"
pasta_reports.mkdir(parents=True, exist_ok=True)

torch.save(modelo.state_dict(), pasta_reports / "mlp_autoencoder_b1.pt")

df_predicoes = df_test[["idx", "filename", "timestamp"]].copy()
df_predicoes["erro_reconstrucao"] = erro_test
df_predicoes["anomalia_detectada"] = (erro_test > limiar).astype(int)
df_predicoes.to_csv(pasta_reports / "predicoes_teste.csv", index=False)

print("Modelo e predições salvos com sucesso em notebooks/reports/:")
display(df_predicoes.head())